# Week 1 · Notebook 2 — Indicators: turning prices into signals

A raw price tells you almost nothing. Is 18.50 high or low? Rising or calm? An
**indicator** is a small calculation over recent prices that answers one such
question with a number. Today you build two, and put them on the chart tool you made
yesterday.

Two functions you build:
1. `sma` — the simple moving average (the trend).
2. `rsi` — the relative strength index (momentum: overbought vs oversold).

## 1. Function — `sma` (simple moving average)

The average of the last `window` prices, recomputed each day. It smooths daily
noise so a trend becomes visible. Because it needs `window` days of history before
it can produce a value, the first `window-1` entries are `NaN`.

**In:** `prices`, `window`. **Out:** an array the same length as `prices`, `NaN`
for the first `window-1` entries.
**Hint:** for each `i`, average `prices[i-window+1 : i+1]`.
**Done when:** the check passes.

In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
from tradinglab.data_feed import DataFeed

feed = DataFeed.from_dir('data/egx'); 
price = feed.close[:, 0]

def sma(prices, window):
    prices = np.asarray(prices, dtype=float)
    out = np.full_like(prices, np.nan)
    # ---8<--- solution
    for i in range(window-1, len(prices)):
        out[i] = prices[i-window+1:i+1].mean()
    # ---8<--- end
    return out

t = sma(np.array([1.,2,3,4,5]), 3)
assert np.isnan(t[:2]).all() and t[2]==2.0 and t[4]==4.0, 'not right yet'
print('sma correct ✓')

## 2. Function — `rsi` (relative strength index)

RSI measures how hard price has been pushed up vs down recently, on a 0–100 scale.
Above ~70 is often called "overbought", below ~30 "oversold". The recipe: average
the up-moves and the down-moves over a window, form `rs = avg_gain / avg_loss`, then
`100 - 100/(1+rs)`.

**In:** `prices`, `window` (default 14). **Out:** array in [0, 100], `NaN` early.
**Hint:** `delta = np.diff(prices)`; gains are positive deltas, losses the absolute
negative ones.
**Done when:** values stay within [0, 100].

In [ ]:
def rsi(prices, window=14):
    prices = np.asarray(prices, dtype=float)
    out = np.full_like(prices, np.nan)
    delta = np.diff(prices)
    gains = np.where(delta > 0, delta, 0.0)
    losses = np.where(delta < 0, -delta, 0.0)
    # ---8<--- solution
    for i in range(window, len(prices)):
        ag = gains[i-window:i].mean(); al = losses[i-window:i].mean()
        out[i] = 100.0 if al == 0 else 100.0 - 100.0/(1.0 + ag/al)
    # ---8<--- end
    return out

r = rsi(price, 14); valid = r[~np.isnan(r)]
assert (valid >= 0).all() and (valid <= 100).all(), 'RSI must be in [0,100]'
print('rsi correct ✓  (recent RSI:', round(np.nanmean(r[-20:]),1), ')')

## 3. See them on your chart
Reuse the `plot_price` tool you built yesterday. Indicators only mean something when
you can see them against price.

In [ ]:
from tradinglab.charting import plot_price   # your graduated tool
ax = plot_price(feed.dates, price,
                overlays={'SMA20': sma(price, 20), 'SMA50': sma(price, 50)},
                title=feed.symbols[0] + ' with moving averages')
plt.show()
print('Notice how the SMAs lag price and smooth the noise — that lag is the trade-off.')

In [ ]:
# --- RSI belongs in its own panel — different scale than price ---
r = rsi(price, 14)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                                gridspec_kw={'height_ratios': [2, 1]})

ax1.plot(feed.dates, price, linewidth=1.2)
ax1.set_title(feed.symbols[0] + ' — price')
ax1.grid(alpha=0.3)

ax2.plot(feed.dates, r, color='purple', linewidth=1.0)
ax2.axhline(70, color='red', linestyle='--', linewidth=0.8, label='overbought (70)')
ax2.axhline(30, color='green', linestyle='--', linewidth=0.8, label='oversold (30)')
ax2.set_ylim(0, 100)
ax2.set_title('RSI(14)')
ax2.legend(loc='upper left', fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Graduate and reflect
You now have `sma` and `rsi`. Move them into `src/tradinglab/indicators.py` and run
`uv run pytest week1/tests/`.

These aren't just charts — tomorrow they become **signals**. When a short SMA rises
above a long one, that's a trend you can trade. That's the strategy you build next.